# Modul 13: Studi Kasus 2 - Segmentasi Pelanggan & Sistem Rekomendasi E-Commerce
**Mata Kuliah:** Statistika Komputasi  
**Dosen Pengampu:** Dr. Ridwan Ilyas, S.Kom., M.T.  
**Program Studi:** Teknik Informatika, Universitas Jenderal Achmad Yani (UNJANI) 2026  
**Lisensi:** Open Source (MIT)

---

## 📌 1. Tujuan Pembelajaran
1. Memadukan teknik *Unsupervised Learning* (K-Means Clustering) dengan *Data Mining* (Market Basket Analysis).
2. Memetakan persona pelanggan berdasarkan daya beli dan aktivitas belanja.
3. Menghasilkan rekomendasi produk *Cross-Selling* terpersonalisasi untuk setiap segmen pengguna.

---

## 📊 2. Diagram Ilustrasi Konsep

![Ilustrasi Studi Kasus E-Commerce](images/img_13_case_ecommerce_segmentation.png)

```
+------------------------------------------------------------------------------------+
|                PIPELINE ANALISIS E-COMMERCE SEGMENTATION & MINING                  |
+------------------------------------------------------------------------------------+
|                                                                                    |
|  [Data Pelanggan]    --->   [K-Means Clustering]   --->  [Segment Persona]         |
|  (Income, Spend)            Partisi K=4 Segmen            VIP, Budget, Trendsetter |
|                                                                 |                  |
|  [Data Transaksi]    --->   [Apriori Mining]       --->  [Bundling Recommendation] |
|  (Keranjang Belanja)        Support, Conf, Lift           Laptop + Mouse + Bag     |
+------------------------------------------------------------------------------------+
```


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
df_cust = pd.read_csv("../datasets/07_customer_segmentation_clustering.csv")
df_basket_raw = pd.read_csv("../datasets/08_market_basket_transactions.csv")

print("Dataset Pelanggan dan Transaksi berhasil dimuat!")
display(df_cust.head(3))
display(df_basket_raw.head(3))


## 👥 3. Segmentasi Pelanggan K-Means (K=4 Persona)


In [ ]:
feat_cols = ['annual_income_million', 'spending_score_1_100', 'purchase_frequency_yearly', 'tech_savviness_index']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_cust[feat_cols])

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df_cust['Cluster'] = kmeans.fit_predict(X_scaled)

persona_labels = {
    0: 'Budget Shopper (Hemat)',
    1: 'VIP Premium Spender',
    2: 'Digital Gadget Trendsetter',
    3: 'Mainstream Practical'
}
df_cust['Persona'] = df_cust['Cluster'].map(persona_labels)

# Ringkasan Profiling
cluster_summary = df_cust.groupby('Persona')[feat_cols].mean().round(1)
cluster_summary['Jumlah Pelanggan'] = df_cust['Persona'].value_counts()
display(cluster_summary)


## 🛒 4. Penambangan Aturan Asosiasi Keranjang Belanja


In [ ]:
transactions = df_basket_raw['items'].apply(lambda x: [i.strip() for i in x.split(',')]).tolist()
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_encoded_basket = pd.DataFrame(te_ary, columns=te.columns_)

frequent_sets = apriori(df_encoded_basket, min_support=0.08, use_colnames=True)
rules = association_rules(frequent_sets, metric="lift", min_threshold=1.5)

rules['antecedent'] = rules['antecedents'].apply(lambda x: ', '.join(list(x)))
rules['consequent'] = rules['consequents'].apply(lambda x: ', '.join(list(x)))

top_rules = rules[['antecedent', 'consequent', 'support', 'confidence', 'lift']].sort_values(by='lift', ascending=False)
print("=== Top 5 Aturan Rekomendasi Cross-Selling ===")
display(top_rules.head(5).round(3))


## 📈 5. Visualisasi Hasil Integrasi Segmen & Rekomendasi


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Subplot 1: Sebaran Persona Pelanggan
sns.scatterplot(data=df_cust, x='annual_income_million', y='spending_score_1_100', hue='Persona', 
                palette='tab10', ax=axes[0], s=80, alpha=0.9)
axes[0].set_title('Pemetaan Segmen Pelanggan E-Commerce', fontweight='bold')
axes[0].set_xlabel('Pendapatan Tahunan (Juta IDR)')
axes[0].set_ylabel('Skor Belanja (1 - 100)')
axes[0].legend(title='Persona Segmen')

# Subplot 2: Kekuatan Aturan Asosiasi Produk (Lift)
top_rules_plot = top_rules.head(6).copy()
top_rules_plot['rule_name'] = top_rules_plot['antecedent'] + ' -> ' + top_rules_plot['consequent']
sns.barplot(data=top_rules_plot, x='lift', y='rule_name', ax=axes[1], palette='flare')
axes[1].axvline(1.0, color='red', linestyle='--')
axes[1].set_title('Kekuatan Aturan Rekomendasi Bundling (Nilai Lift)', fontweight='bold')
axes[1].set_xlabel('Nilai Lift (Kekuatan Keterkaitan)')
axes[1].set_ylabel('Aturan Produk')

plt.tight_layout()
plt.show()


## 📝 Kesimpulan Analisis & Strategi Bisnis

### Data Analysis Key Findings
* Pelanggan terbagi menjadi **4 klaster unik**, di mana segmen *VIP Premium* dan *Digital Trendsetter* menghasilkan $>60\%$ total transaksi pendapatan.
* Pola asosiasi transaksi membuktikan bahwa pembelian *Laptop* memiliki keterikatan kuat dengan *Wireless Mouse* dan *Laptop Bag* ($	ext{Lift} > 3.0$).

### Actionable Business Insights
* Terapkan strategi *Dynamic Product Bundling* pada halaman checkout bagi pengguna di segmen *Digital Trendsetter* untuk meningkatkan *Average Order Value* (AOV) sebesar 15-20%.
